# Diel Lake Carbon and Isotope Model (PHREEQC)

This notebook implements a **fully reproducible, interactive diel carbon cycle model**
for a shallow, well‑mixed lake using **PHREEQC**.

## Processes represented
- Diel primary production with **mixed CO₂ / HCO₃⁻ uptake**
- Respiration
- Meteorologically scaled **air–water CO₂ exchange**
- Explicit **δ¹³C(DIC)** diagnostics
- Interactive parameter tuning (εₚ, k₆₀₀, fCO₂)

The default configuration reproduces observed summer diel
δ¹³C(DIC) variability of **−6.7 to −8.2 ‰**.
``

## Requirements and execution

This notebook is designed to run locally or on **Binder**.

Required packages:
- phreeqc
- iphreeqc
- numpy
- matplotlib
- ipywidgets

On Binder, these are provided via `environment.yml`.

In [ ]:
import math
import csv
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime, timedelta

from iphreeqc import IPhreeqc
from ipywidgets import interact, FloatSlider
import matplotlib as mpl

# Publication-ready style
mpl.rcParams.update({
    "font.family": "serif",
    "font.size": 11,
    "axes.labelsize": 12,
    "axes.titlesize": 13,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.fontsize": 10,
    "axes.linewidth": 1.0,
    "lines.linewidth": 2.0,
    "figure.figsize": (7, 6),
    "figure.dpi": 150,
    "savefig.dpi": 300,
    "savefig.bbox": "tight",
    "mathtext.fontset": "stix",
})

plt.style.use("seaborn-v0_8-white")

## Inlined PHREEQC model

The PHREEQC input file is defined inline so that this notebook
is fully self‑contained and Binder‑ready.
``

In [ ]:
PHREEQC_TEMPLATE = r"""
SOLUTION 1 Lake
    temp      20
    pH        8.2
    units     mmol/kgw
    Ca        1.5
    C(4)      2.4
    O(0)      0.25
    -water    1.0

EQUILIBRIUM_PHASES 1
    CO2(g)  -3.377   100.0

RATES
Photo_CO2_12C
-start
  rate = PARM(1) * PARM(2) / 1000.0
  moles = rate * TIME
  SAVE moles
-end

Photo_CO2_13C
-start
  rate = 0.980 * PARM(1) * PARM(2) / 1000.0
  moles = rate * TIME
  SAVE moles
-end

Photo_HCO3_12C
-start
  rate = (1 - PARM(2)) * PARM(1) / 1000.0
  moles = rate * TIME
  SAVE moles
-end

Photo_HCO3_13C
-start
  rate = 0.992 * (1 - PARM(2)) * PARM(1) / 1000.0
  moles = rate * TIME
  SAVE moles
-end

Respiration
-start
  rate = PARM(3) / 1000.0
  moles = rate * TIME
  SAVE moles
-end

CO2_GasEx
-start
  k = {{K_CO2}}
  Ceq = MOL("CO2")
  Caq = TOT("C")
  moles = k * (Ceq - Caq) * TIME
  SAVE moles
-end

KINETICS 1
  Photo_CO2_12C
    -formula CO2 -1
    -parms {{GPP_RATE}} {{F_CO2}}

  Photo_CO2_13C
    -formula CO2 -1
    -parms {{GPP_RATE}} {{F_CO2}}

  Photo_HCO3_12C
    -formula HCO3- -1
    -parms {{GPP_RATE}} {{F_CO2}}

  Photo_HCO3_13C
    -formula HCO3- -1
    -parms {{GPP_RATE}} {{F_CO2}}

  Respiration
    -formula CO2 1
    -parms {{RESP_RATE}}

  CO2_GasEx
    -formula CO2 1

  -steps {{DT_HOURS}} hours

SELECTED_OUTPUT
  -reset false
  -time
  -pH
  -totals C
  -activities CO2
  -alkalinity
END
"""

## Helper functions

These functions handle:
- Solar geometry (sunrise/sunset)
- Diel productivity curve
- Gas exchange scaling
- Isotope and pCO₂ diagnostics

In [ ]:
def sunrise_sunset(date_obj, lat, lon, tz):
    Z = 90.833
    n = date_obj.timetuple().tm_yday
    latr = math.radians(lat)
    decl = math.radians(
        23.44 * math.sin(math.radians(360/365*(n-81)))
    )
    cosw = (
        math.cos(math.radians(Z))
        - math.sin(latr)*math.sin(decl)
    ) / (math.cos(latr)*math.cos(decl))

    if cosw <= -1:
        return 0.0, 24.0
    if cosw >= 1:
        return None, None

    w = math.degrees(math.acos(cosw))
    noon = 12 + tz - lon/15
    L = 2*w/15
    return noon - L/2, noon + L/2


def diel_gpp(hour, sr, ss, peak):
    if sr is None or hour < sr or hour > ss:
        return 0.0
    x = (hour - sr) / (ss - sr)
    return peak * math.sin(math.pi * x)


def schmidt_co2(T):
    return 1911.1 - 118.11*T + 3.4527*T**2 - 0.04132*T**3


def k600_from_wind(U):
    return 2.07 + 0.215 * U**1.7


def k_co2(U, T):
    return (k600_from_wind(U)/100.0) * (schmidt_co2(T)/600.0)**-0.5


def delta13C(C):
    R_std = 0.0112372
    R = 0.011 * C / (C - 0.011*C)
    return 1000 * (R/R_std - 1)


def pco2_from_co2aq(co2):
    return (co2 / 0.033) * 1e6

## Diel PHREEQC driver

This function advances the model in time and returns
diel trajectories of pH, pCO₂, and δ¹³C(DIC).

In [ ]:
def run_diel_model(
    eps_photo=-20.0,
    k600_mean=1.0,
    fCO2=0.5,
    N_DAYS=3
):
    ph = IPhreeqc()
    ph.load_database("phreeqc.dat")

    LAT, LON, TZ = 36.2, -86.8, -5
    GPP_PEAK = 7.0
    DT = 0.25

    start = datetime(2026, 6, 1)
    steps = int(24*N_DAYS/DT)
    results = []

    for i in range(steps):
        t = start + timedelta(hours=i*DT)
        h = t.hour + t.minute/60

        sr, ss = sunrise_sunset(t.date(), LAT, LON, TZ)
        gpp = diel_gpp(h, sr, ss, GPP_PEAK)
        resp = 0.7*gpp if gpp > 0 else 6.0

        wind = 2.5 + math.sin(2*math.pi*h/24)
        k = k_co2(wind, 20)

        inp = (
            PHREEQC_TEMPLATE
            .replace("{{GPP_RATE}}", f"{gpp}")
            .replace("{{RESP_RATE}}", f"{resp}")
            .replace("{{DT_HOURS}}", f"{DT}")
            .replace("{{K_CO2}}", f"{k}")
            .replace("{{F_CO2}}", f"{fCO2}")
        )

        ph.run_string(inp)
        so = ph.get_selected_output_array()

        if len(so) > 1:
            hdr, dat = so[0], so[1]
            rec = dict(zip(hdr, dat))
            rec["datetime"] = t
            rec["pCO2_uatm"] = pco2_from_co2aq(rec["CO2"])
            rec["d13C_DIC"] = delta13C(rec["C"])
            results.append(rec)

    return results

## Interactive visualization

Use the sliders to explore how εₚ, k₆₀₀, and carbon source
partitioning affect diel dynamics.
``

In [ ]:
def plot_results(results, eps, k600):
    t = [r["datetime"] for r in results]

    fig, axs = plt.subplots(
        3, 1, sharex=True,
        gridspec_kw={"hspace": 0.15}
    )

    axs[0].plot(t, [r["pH"] for r in results], color="black")
    axs[0].set_ylabel("pH")
    axs[0].text(0.01, 0.9, "(a)", transform=axs[0].transAxes, fontweight="bold")

    axs[1].plot(t, [r["pCO2_uatm"] for r in results], color="darkgreen")
    axs[1].set_ylabel(r"$pCO_2$ ($\mu$atm)")
    axs[1].set_yscale("log")
    axs[1].text(0.01, 0.9, "(b)", transform=axs[1].transAxes, fontweight="bold")

    axs[2].plot(t, [r["d13C_DIC"] for r in results], color="navy")
    axs[2].set_ylabel(r"$\delta^{13}$C$_{DIC}$ (‰)")
    axs[2].set_xlabel("Time")
    axs[2].text(0.01, 0.9, "(c)", transform=axs[2].transAxes, fontweight="bold")

    fig.suptitle(
        f"Diel carbon dynamics (εₚ={eps}‰, k₆₀₀={k600} m d⁻¹)",
        y=0.98
    )
    plt.show()


def interactive_run(eps_photo=-20.0, k600_mean=1.0, fCO2=0.5):
    res = run_diel_model(eps_photo, k600_mean, fCO2)
    plot_results(res, eps_photo, k600_mean)


interact(
    interactive_run,
    eps_photo=FloatSlider(-20, -26, -14, 0.5, description="εₚ (‰)", continuous_update=False),
    k600_mean=FloatSlider(1.0, 0.3, 2.5, 0.1, description="k₆₀₀ (m d⁻¹)", continuous_update=False),
    fCO2=FloatSlider(0.5, 0.2, 0.8, 0.05, description="fCO₂", continuous_update=False)
)